# Shor's Algorithm from Scratch, Part 3: Controlled Modular Multiplication

In [Part 1](401_shor_scratch_adder.ipynb) we built an ordinary adder, and in [Part 2](402_shor_scratch_modulo_adder.ipynb) we built modular addition $a+b \bmod N$. This time we implement **controlled modular multiplication**. Given a fixed classical constant $a$, we target equation (11) of the paper directly.

$$
\ket{c; x, 0} \longrightarrow
\begin{cases}
\ket{c; x,\ a x \bmod N} & (\ket c = \ket 1) \\
\ket{c; x,\ x} & (\ket c = \ket 0)
\end{cases}
$$

That is, we multiply $x$ by the constant $a$ and take the remainder modulo $N$, but this happens only when an additional control qubit $c$ is $1$; when $c=0$, $x$ is simply copied into the result register. This is exactly the operation we will use repeatedly to build the **modular exponentiation** in the next part: $a^j \bmod N$ is a chain of controlled multiplications by $a, a^2, a^4, a^8, \ldots$, each controlled by a bit of $j$.

Why the $c=0$ branch gives $\ket x$ rather than $\ket 0$ is explained at the end of this notebook.

Reference: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995), Figure 5

## Idea: multiplication is repeated conditional addition

Multiplication by longhand in binary means: for each bit $x_i$ of $x$, if $x_i=1$ then add $a \cdot 2^i$ to the total. The paper says the same thing.

> Function $f_{a,N}(x) = ax \bmod N$ can be implemented by repeated conditional additions (modulo $N$).

By performing this process **taking the remainder modulo $N$ every time** (not just once at the end), we keep all intermediate values in range. This matters because the adder registers are designed to hold only values less than $N$.

So the circuit is: for $i = 0, \ldots, n-1$, use the modular adder from Part 2 to add the *constant* $(a \cdot 2^i) \bmod N$ into the result register $b$ — but only on branches of the superposition where **both** the multiplier's own control qubit $c$ **and** $x_i$ are $1$.

The problem is how to implement "only when two qubits are both 1". The trick we saw in Step 4 of Part 2 works here as well.

In [ ]:
!pip install git+https://github.com/blueqat/blueqatSDK

## The trick from Fig.5: control the "constant", not the adder

The naive approach would be to **add two controls to every gate** of the modular adder from Part 2. But then gates that were at most 2-controlled (Toffoli) in Part 2 become 4-controlled, requiring decomposition of multi-controlled gates, and the circuit blows up rapidly.

The paper does not do this.

> To account for this fact at the $i$th modular addition stage the first register is loaded with the value $2^i a$ if $\ket{c, x_i} = \ket{1,1}$ and with value 0 otherwise. This is done by applying the Toffoli gate to the control qubits $\ket c$ and $\ket{x_i}$ and the appropriate target qubit in the register; the gate is applied each time value "1" appears in the binary form of the number $2^i a$.

In other words, **we do not touch the adder at all**. Instead, we make the write into the work register (hereafter `const`) that holds the addend conditional: load $2^i a$ only when $(c, x_i) = (1,1)$, and **0** otherwise.

Why is that enough? **Because modular addition of $0$ does nothing.** If $b < N$ then $(0 + b) \bmod N = b$. The adder itself can run unconditionally; if the input is $0$, it becomes the identity transformation on its own.

This is exactly the same idea as Step 4 of Part 2 (conditionally zeroing the $N$ register). The paper uses this trick in two places, Fig.4 and Fig.5. **Rather than making the operation itself conditional, make the input conditional so that the operation becomes trivial.**

And undoing the load is easy too.

> Resetting the register to its initial state is done by applying the same sequence of the Toffoli gates again (the order of the gates is irrelevant as they act on different target qubits).

Since Toffoli is self-inverse, **running the same column again** unloads it.

As a consequence of this approach, **no gate beyond Toffoli (`CCX`) ever appears in this notebook's circuit**. No decomposition of multi-controlled gates and no ancilla qubits for that purpose are needed.

## Reusing the parts from Part 2 as-is

We use the modular adder from Part 2 as-is (the version that conditionally zeroes the $N$ register, exactly as in Fig.4 of the paper).

In [2]:
from blueqat import Circuit

def carry(c, a, b, c_next):
    circ = Circuit()
    circ.ccx[a, b, c_next]
    circ.cx[a, b]
    circ.ccx[c, b, c_next]
    return circ

def carry_inv(c, a, b, c_next):
    circ = Circuit()
    circ.ccx[c, b, c_next]
    circ.cx[a, b]
    circ.ccx[a, b, c_next]
    return circ

def sum_gate(c, a, b):
    circ = Circuit()
    circ.cx[a, b]
    circ.cx[c, b]
    return circ

def add_circuit(c, a, b, n):
    """b := a + b, using carry register c. b must have n+1 qubits."""
    circ = Circuit()
    for i in range(n - 1):
        circ += carry(c[i], a[i], b[i], c[i + 1])
    circ += carry(c[n - 1], a[n - 1], b[n - 1], b[n])
    circ.cx[a[n - 1], b[n - 1]]
    circ += sum_gate(c[n - 1], a[n - 1], b[n - 1])
    for i in range(n - 2, -1, -1):
        circ += carry_inv(c[i], a[i], b[i], c[i + 1])
        circ += sum_gate(c[i], a[i], b[i])
    return circ

def sub_circuit(c, a, b, n):
    """b := b - a: the adder's gate list, reversed."""
    add_circ = add_circuit(c, a, b, n)
    reversed_circ = Circuit()
    reversed_circ.ops = list(reversed(add_circ.ops))
    return reversed_circ

def set_to_zero_if(t, N, val_N, n):
    """Reset the N register to 0 when t == 0. Self-inverse."""
    circ = Circuit()
    circ.x[t]
    for j in range(n):
        if (val_N >> j) & 1:
            circ.cx[t, N[j]]
    circ.x[t]
    return circ

def modulo_add_circuit(c, a, b, N, t, n, val_N):
    """b := (a + b) mod N, on the given registers. Paper Fig.4."""
    circ = Circuit()
    circ += add_circuit(c, a, b, n)          # 1. b += a
    circ += sub_circuit(c, N, b, n)          # 2. b -= N
    circ.cx[b[n], t]                         # 3. overflow -> t
    gate = set_to_zero_if(t, N, val_N, n)
    circ += gate                             # 4. blank N when t == 0 ...
    circ += add_circuit(c, N, b, n)          #    ... plain adder ...
    circ += gate                             #    ... restore N
    circ += sub_circuit(c, a, b, n)          # 5. clean up
    circ.x[b[n]]
    circ.cx[b[n], t]
    circ.x[b[n]]
    circ += add_circuit(c, a, b, n)
    return circ

## Controlled multiplier

There are $5n+3$ qubits in total.

- `c[0..n-1]`: carry register
- `const[0..n-1]`: work register. Holds $(a \cdot 2^i) \bmod N$ at each stage
- `b[0..n]`: result register ($n+1$ wide)
- `N[0..n-1]`: modulus
- `x[0..n-1]`: the number being multiplied
- `ctrl`: the multiplier's control qubit $c$
- `t`: the modular adder's overflow flag

For each bit $i$ of $x$:

1. Load the classical constant $(a \cdot 2^i) \bmod N$ into `const` using Toffolis controlled by `ctrl` and $x_i$
2. Run the **uncontrolled** modular adder
3. Run the same Toffoli column again to reset `const` to $\ket 0$

The reason we can compute the classical constant at circuit-construction time is that $a$, $N$, and $i$ are all known constants and do not depend on the superposition.

Finally, we add the "copy $x$ if $c=0$" part of equation (11) (the rightmost Toffoli column in Fig.5 of the paper).

In [3]:
def build_controlled_multiplier(mult_a, val_x, val_N, ctrl_value, n):
    c     = list(range(n))                     # carry register
    const = list(range(n, 2 * n))              # scratch: holds (a * 2**i) mod N
    b     = list(range(2 * n, 3 * n + 1))      # result register (n+1 wide)
    N     = list(range(3 * n + 1, 4 * n + 1))  # modulus
    x     = list(range(4 * n + 1, 5 * n + 1))  # multiplicand
    ctrl  = 5 * n + 1                          # multiplier's control qubit
    t     = 5 * n + 2                          # modulo-adder's overflow flag
    circ = Circuit(5 * n + 3)                  # no ancilla needed

    if ctrl_value:
        circ.x[ctrl]
    for i in range(n):
        if (val_x >> i) & 1:
            circ.x[x[i]]
        if (val_N >> i) & 1:
            circ.x[N[i]]

    # The adder carries no extra controls, and is identical at every stage,
    # so it can be built once outside the loop and reused.
    adder = modulo_add_circuit(c, const, b, N, t, n, val_N)

    for i in range(n):
        const_val = (mult_a * (1 << i)) % val_N
        load = Circuit()
        for j in range(n):
            if (const_val >> j) & 1:
                load.ccx[ctrl, x[i], const[j]]   # <- the only controlled part
        circ += load      # load 2^i * a if (c, x_i) == (1, 1), else 0
        circ += adder     # b := (const + b) mod N
        circ += load      # Toffolis are self-inverse -> unload

    # eq.(11): if ctrl == 0, copy x into the result register
    circ.x[ctrl]
    for i in range(n):
        circ.ccx[ctrl, x[i], b[i]]
    circ.x[ctrl]

    return circ, b

In [4]:
def run_controlled_multiplier(mult_a, val_x, val_N, ctrl_value, n=None, quiet=False):
    """a, x, N, ctrl can be any values you like -- n (register width) is inferred
    from N and x automatically if not given explicitly."""
    if n is None:
        n = max(val_N, val_x, 1).bit_length()
    circuit, b = build_controlled_multiplier(mult_a, val_x, val_N, ctrl_value, n)
    total = circuit.n_qubits
    if not quiet:
        print(f"  [n={n}, qubits={total}, gates={len(circuit.ops)}]")
    state = circuit.m[:].run(shots=1, backend="statevector").most_common(1)[0][0]
    bits = [state[total - 1 - q] for q in b]
    return int("".join(reversed(bits)), 2)

for a, x, N, ctrl in [(2, 1, 3, 1), (2, 2, 3, 1), (2, 1, 3, 0), (2, 2, 3, 0)]:
    result = run_controlled_multiplier(a, x, N, ctrl)
    expected = (a * x) % N if ctrl else x        # <- eq.(11)
    print(f"ctrl={ctrl}: {a}*{x} mod {N} = {result}  (expected {expected}, "
          f"{'OK' if result == expected else 'MISMATCH'})")

  [n=2, qubits=13, gates=176]


ctrl=1: 2*1 mod 3 = 2  (expected 2, OK)
  [n=2, qubits=13, gates=176]


ctrl=1: 2*2 mod 3 = 1  (expected 1, OK)
  [n=2, qubits=13, gates=175]


ctrl=0: 2*1 mod 3 = 1  (expected 1, OK)
  [n=2, qubits=13, gates=175]


ctrl=0: 2*2 mod 3 = 2  (expected 2, OK)


With `ctrl=1` the multiplication is performed, and with `ctrl=0` $x$ appears as-is in the result register. This matches equation (11).

## Exhaustive test

For $N=7$, we try all values of $x$ with both values of `ctrl`. Since the circuit is linear, if it is correct on every computational-basis input, it also works correctly on superpositions of them.

In [5]:
bad = 0
for ctrl in (0, 1):
    for x in range(7):
        result = run_controlled_multiplier(3, x, 7, ctrl, n=3, quiet=True)
        expected = (3 * x) % 7 if ctrl else x
        if result != expected:
            bad += 1
            print(f"MISMATCH ctrl={ctrl} x={x}: {result} != {expected}")
print(f"14 cases checked, {bad} mismatches")

14 cases checked, 0 mismatches


## A note on scale

This circuit can be assembled with ordinary Python code alone (no quantum superposition required), so we can display the exact qubit and gate counts before running it. Let us look up to $N=15$ — the classic first example of Shor's algorithm.

In [6]:
for a, x, N, n in [(2, 2, 3, 2), (3, 5, 7, 3), (7, 13, 15, 4)]:
    circ, _ = build_controlled_multiplier(a, x, N, 1, n)
    print(f"N={N:2d} (n={n}): {circ.n_qubits:2d} qubits, {len(circ.ops):4d} gates")

N= 3 (n=2): 13 qubits,  176 gates
N= 7 (n=3): 18 qubits,  395 gates
N=15 (n=4): 23 qubits,  702 gates


The number of qubits grows only **linearly** in $n$, as $5n+3$. The cost of classically simulating a single shot, however, does not. Measured on the author's machine:

| | Qubits | Gates | Time per shot |
|---|---|---|---|
| $N=3$ | 13 | 176 | 0.3 s |
| $N=7$ | 18 | 395 | 5.4 s |
| $N=15$ | 23 | 702 | **2047 s** |

In state-vector simulation, the cost per gate is proportional to the number of amplitudes, i.e. $2^{(\text{number of qubits})}$. Going from 18 to 23 qubits multiplies the amplitudes by $2^5 = 32$, and multiplying by the 1.8× gate count gives naively **60×**. The measured value was **380×** (small circuits are dominated by Python's per-gate overhead, while large circuits become memory-bandwidth-bound, so the estimate worsens as circuits grow).

This is not to say that blueqat is slow. Building the circuit finishes in polynomial time in $n$, but classically mimicking it gets hit by an exponential — this is a small hands-on taste of **why factoring is believed to be classically hard**. On real hardware, the cost of running the circuit has nothing to do with the *simulator's* constraints.

If you want to actually run $N=15$, uncomment the next cell (it takes about 34 minutes).

In [7]:
# result = run_controlled_multiplier(7, 13, 15, 1, n=4)   # 7 * 13 mod 15 = 1
# print(result)

## Why $\ket x$ rather than $\ket 0$ when $c=0$

Finally, we tie up the second half of equation (11).

What this multiplier does is $U_{a,N}\ket{x, 0} \to \ket{x, ax \bmod N}$, with **the input $x$ left in place**. What Shor's algorithm really wants is a map that rewrites the same register as $\ket x \to \ket{ax \bmod N}$. The paper achieves this by first computing $a^{-1} \bmod N$ classically (efficiently computable via the Euclidean algorithm), then

1. $U_{a,N}$ maps $\ket{x, 0} \to \ket{x,\ ax}$
2. **Swap** the two registers to get $\ket{ax,\ x}$
3. Apply the **inverse** of $U_{a^{-1},N}$ to get $\ket{ax,\ 0}$

The reason step 3 clears the register is that $U_{a^{-1},N}\ket{ax, 0} = \ket{ax,\ a^{-1}ax} = \ket{ax,\ x}$.

Now consider the case $c = 0$. If the control is $0$, the whole thing should be the identity, i.e. it should come out as $\ket{x,0}$.

- If the result register stayed $\ket 0$ when $c=0$, the swap in step 2 would give $\ket{0,\ x}$. $x$ would go to the other side and never come back
- If it becomes $\ket{x,x}$ when $c=0$, it stays $\ket{x,x}$ after the swap. The inverse operation in step 3 restores the second register to $\ket 0$, happily giving $\ket{x,0}$

Thanks to the copy, **the swap itself can be uncontrolled**. The seemingly mysterious second half of equation (11) was a setup for the next construction.

## Next time

Now that we have controlled multiplication, the next component is [modular exponentiation](404_shor_scratch_modular_exponentiation.ipynb) — using this circuit as the core, we repeatedly square while controlling on each bit of the phase-estimation register. The register swap and the inverse operation via $a^{-1} \bmod N$ described above will actually appear there.

This is the operation that the quantum part of Shor's algorithm actually executes, and it is the component that ultimately connects this whole series to the order-finding theory in [400_shor.ipynb](400_shor.ipynb).